# Stage 3 — DPO Preference Alignment

Preference alignment of the Stage 2 instruction-tuned `Qwen/Qwen2.5-0.5B` on California homeowners-insurance chosen/rejected pairs using [Unsloth](https://github.com/unslothai/unsloth) + `trl`'s `DPOTrainer`, LoRA/QLoRA, on a free Colab T4.

Spec: `specs/002-unsloth-ca-homeowners-finetune.md` §6-7 (Stage 3).

**Run this notebook on Google Colab with a T4 GPU runtime** (Runtime → Change runtime type → T4 GPU). It has not been executed locally — there is no CUDA GPU in the local dev environment for this repo.

Before running: upload/clone this repo into the Colab session (`!git clone <repo-url>` then `%cd <repo-name>`), and run Stage 2 (`notebooks/2-instruction_finetuning.ipynb`) first so it has pushed its merged model to the Hugging Face Hub — this notebook pulls that checkpoint directly (`HF_STAGE2_REPO`), no manual upload needed. Add an `HF_TOKEN` Colab secret before running, and set `HF_USERNAME` in the paths cell to your Hugging Face username. This stage's own output (`HF_STAGE3_REPO`) is the final model — that's what `src/inference.py` will load.

In [1]:
!git clone https://github.com/saravanangenai/LLM-FineTunning.git
%cd LLM-FineTunning/notebooks

Cloning into 'LLM-FineTunning'...
remote: Enumerating objects: 117, done.
remote: Counting objects: 100% (117/117), done.
remote: Compressing objects: 100% (91/91), done.
remote: Total 117 (delta 63), reused 69 (delta 25), pack-reused 0 (from 0)
Receiving objects: 100% (117/117), 189.97 KiB | 7.04 MiB/s, done.
Resolving deltas: 100% (63/63), done.
/content/LLM-FineTunning/notebooks


## 0. Install dependencies

In [2]:
%%capture
import importlib.util

if importlib.util.find_spec("unsloth") is None:
    !pip install unsloth
    !pip install --no-deps "trl>=0.9.0" "peft>=0.11.0" "accelerate>=0.30.0" "bitsandbytes>=0.43.0"

In [3]:
from google.colab import userdata
from huggingface_hub import login

HF_TOKEN = userdata.get("HF_TOKEN")
login(token=HF_TOKEN)

In [4]:
import sys
from pathlib import Path

# Assumes this notebook is run from notebooks/ inside the cloned repo, so the
# repo root (one level up) contains the src/ package and data/ directory.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

DATA_PATH = REPO_ROOT / "data" / "3-preference_dataset.jsonl"
STAGE2_MODEL_DIR = REPO_ROOT / "models" / "stage2-merged"
ADAPTER_OUT_DIR = REPO_ROOT / "models" / "stage3-dpo-adapter"
MERGED_OUT_DIR = REPO_ROOT / "models" / "stage3-final-merged"
print("Repo root:", REPO_ROOT)
print("Data path exists:", DATA_PATH.exists())
print("Stage 2 merged model exists:", STAGE2_MODEL_DIR.exists())

Repo root: /content/LLM-FineTunning
Data path exists: True
Stage 2 merged model exists: False


## 2. Load the Stage 2 model (continuing from Stage 2)

Loads the Stage 2 **merged** checkpoint (base weights with Stage 1 domain adaptation + Stage 2 instruction tuning baked in) as this stage's starting "base" model, then attaches a fresh set of Stage 3 LoRA adapters below (spec §6.4). Pulled from the Hugging Face Hub (`HF_STAGE2_REPO`) unless a local copy already exists in this session (e.g. Stage 2 ran in this same runtime).

In [5]:
from src.data_utils import load_jsonl

pairs = load_jsonl(DATA_PATH)
print(f"Loaded {len(pairs)} chosen/rejected pairs")
print(pairs[0])

Loaded 63 chosen/rejected pairs
{'prompt': 'What does dwelling coverage mean in a homeowners policy?', 'chosen': "Dwelling coverage, or Coverage A, pays to repair or rebuild the physical structure of your home, including the roof, walls, and attached structures, after a covered loss. The limit is based on estimated rebuild cost, not market value, so it's worth reviewing at each renewal as construction costs rise.", 'rejected': "Dwelling coverage covers your dwelling. It's the part of the policy about the house."}


In [6]:
from src.model_utils import BASE_MODEL_NAME, MAX_SEQ_LENGTH, load_base_model

stage2_source = STAGE2_MODEL_DIR if STAGE2_MODEL_DIR.exists() else BASE_MODEL_NAME
if stage2_source == BASE_MODEL_NAME:
    print("Stage 2 merged model not found locally — falling back to the raw base model.")

model, tokenizer = load_base_model(
    model_name=str(stage2_source),
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)
print("Loaded:", stage2_source, "| eos_token:", tokenizer.eos_token)

Stage 2 merged model not found locally — falling back to the raw base model.
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.7.3: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loaded: Qwen/Qwen2.5-0.5B | eos_token: <|endoftext|>


## 3. Format the preference dataset for `DPOTrainer`

In [7]:
from src.data_utils import build_preference_dataset

dataset = build_preference_dataset(pairs, tokenizer)
print(dataset)
print(dataset[0]["prompt"][:300])

Dataset({
    features: ['prompt', 'chosen', 'rejected'],
    num_rows: 63
})
<|im_start|>system
You are a helpful assistant that answers general questions about California homeowners insurance, including HO-3 policies, the CA FAIR Plan, and related topics. Answers are for general educational purposes only, not licensed insurance advice or a binding coverage determination.<|i


## 4. Apply fresh LoRA / QLoRA adapters for Stage 3

The same PEFT model is used as both the policy and (with adapters disabled) the implicit reference model — passing `ref_model=None` to `DPOTrainer` below relies on this.

In [8]:
from src.model_utils import add_lora_adapters

model = add_lora_adapters(model)  # uses DEFAULT_LORA_CONFIG: r=16, alpha=16, dropout=0.05

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.7.3 patched 24 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


## 5. Configure and run `DPOTrainer`

In [9]:
from trl import DPOConfig, DPOTrainer

dpo_args = DPOConfig(
    output_dir=str(REPO_ROOT / "outputs" / "stage3"),
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,  # effective batch size 4, per spec §6.3
    num_train_epochs=2,
    learning_rate=5e-6,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    optim="adamw_8bit",
    beta=0.1,
    max_length=MAX_SEQ_LENGTH,
    max_prompt_length=512,
    logging_steps=5,
    save_strategy="no",
    report_to="none",
    seed=42,
)

dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=dpo_args,
    train_dataset=dataset,
    tokenizer=tokenizer,
)

dpo_result = dpo_trainer.train()
print(dpo_result.metrics)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Extracting prompt in train dataset (num_proc=6):   0%|          | 0/63 [00:00<?, ? examples/s]

Applying chat template to train dataset (num_proc=6):   0%|          | 0/63 [00:00<?, ? examples/s]

Tokenizing train dataset (num_proc=6):   0%|          | 0/63 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 63 | Num Epochs = 2 | Total steps = 32
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 2 x 1) = 4
 "-____-"     Trainable parameters = 8,798,208 of 502,830,976 (1.75% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss,rewards / chosen,rewards / rejected,rewards / accuracies,rewards / margins,logps / chosen,logps / rejected,logits / chosen,logits / rejected
5,0.693118,0.000623,0.000564,0.100000,0.000059,-223.408127,-96.042557,-0.617914,-0.160310
10,0.679268,0.043410,0.015327,0.900000,0.028082,-218.525909,-89.365372,-0.748886,-0.469786
15,0.640413,0.126644,0.017705,1.000000,0.108939,-204.069046,-88.371971,-0.682536,-0.262349
20,0.596906,0.244289,0.039920,1.000000,0.204368,-217.335037,-81.012131,-0.800575,-0.394434
25,0.566792,0.326886,0.050648,0.950000,0.276238,-223.464066,-93.785927,-0.524440,-0.201128
30,0.559517,0.359751,0.065233,0.950000,0.294518,-207.525269,-94.675339,-0.671749,-0.217640


{'train_runtime': 44.1936, 'train_samples_per_second': 2.851, 'train_steps_per_second': 0.724, 'total_flos': 0.0, 'train_loss': 0.6180791184306145, 'epoch': 2.0}


## 6. Save the final Stage 3 adapter and merged model

Colab local disk is not persistent across sessions — copy these directories to Google Drive or push them to a private Hugging Face model repo after this cell (see spec §6.4). `MERGED_OUT_DIR` is the final model `src/inference.py` loads.

In [ ]:
ADAPTER_OUT_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(ADAPTER_OUT_DIR))
tokenizer.save_pretrained(str(ADAPTER_OUT_DIR))
print("Saved adapter to:", ADAPTER_OUT_DIR)

Also save a full merged (base weights + LoRA) 16-bit checkpoint to `MERGED_OUT_DIR` — this is the final model that `src/inference.py` loads.

In [ ]:
from src.model_utils import save_merged_model

MERGED_OUT_DIR.mkdir(parents=True, exist_ok=True)
save_merged_model(model, tokenizer, str(MERGED_OUT_DIR))
print("Saved merged model to:", MERGED_OUT_DIR)

In [11]:
HF_USERNAME = "sharanmini"  # <-- set this to your Hugging Face username
HF_STAGE3_REPO = f"{HF_USERNAME}/qwen2.5-0.5b-ca-homeowners-final"

# Persist Stage 3 (final) outputs by pushing them to the Hugging Face Hub —
# this avoids both Colab's occasionally-flaky Drive-mount OAuth flow and a
# slow browser upload/download round trip. HF_STAGE3_REPO is the final
# model — that's what src/inference.py loads.
from src.model_utils import push_folder_to_hub

push_folder_to_hub(str(ADAPTER_OUT_DIR), HF_STAGE3_REPO + "-adapter", token=HF_TOKEN)
push_folder_to_hub(str(MERGED_OUT_DIR), HF_STAGE3_REPO, token=HF_TOKEN)
print("Pushed final model to:", f"https://huggingface.co/{HF_STAGE3_REPO}")

# Alternative: download as a zip instead of using the Hub (no HF token needed).
# import shutil
# from google.colab import files
# shutil.make_archive("/content/stage3-final-merged", "zip", str(MERGED_OUT_DIR))
# files.download("/content/stage3-final-merged.zip")
# shutil.make_archive("/content/stage3-dpo-adapter", "zip", str(ADAPTER_OUT_DIR))
# files.download("/content/stage3-dpo-adapter.zip")

# Alternative: persist to Google Drive instead (can hit a flaky OAuth error).
# from google.colab import drive
# drive.mount("/content/drive")
# shutil.copytree(ADAPTER_OUT_DIR, "/content/drive/MyDrive/stage3-dpo-adapter", dirs_exist_ok=True)
# shutil.copytree(MERGED_OUT_DIR, "/content/drive/MyDrive/stage3-final-merged", dirs_exist_ok=True)

HfHubHTTPError: (Request ID: Root=1-6a5ad60d-5ad15c3411f6b5e2752e6598;71620e32-e3a2-4459-854a-474a56291f23)

403 Forbidden: You don't have the rights to create a model under the namespace "your-hf-username".
Cannot access content at: https://huggingface.co/api/repos/create.
Make sure your token has the correct permissions.

In [ ]:
# Persist Stage 3 (final) outputs by downloading them as a zip — this avoids
# Colab's occasionally-flaky Drive-mount OAuth flow ("credential propagation
# was unsuccessful"). `stage3-final-merged.zip` is the final model —
# download it locally for use with `src/inference.py`.
import shutil

from google.colab import files

shutil.make_archive("/content/stage3-final-merged", "zip", str(MERGED_OUT_DIR))
files.download("/content/stage3-final-merged.zip")

shutil.make_archive("/content/stage3-dpo-adapter", "zip", str(ADAPTER_OUT_DIR))
files.download("/content/stage3-dpo-adapter.zip")

# Alternative: persist to Google Drive instead (can hit the same OAuth error above).
# from google.colab import drive
# drive.mount("/content/drive")
# shutil.copytree(ADAPTER_OUT_DIR, "/content/drive/MyDrive/stage3-dpo-adapter", dirs_exist_ok=True)
# shutil.copytree(MERGED_OUT_DIR, "/content/drive/MyDrive/stage3-final-merged", dirs_exist_ok=True)

## 7. Run the 10 fixed evaluation questions

Same fixed question set as `reports/base_model_evaluation.md` and `reports/sft_model_comparison.md` (spec §4/§8), so answers are directly comparable. Copy these into `reports/final_evaluation.md` alongside the base and Stage 2 (SFT) answers.

In [ ]:
from unsloth import FastLanguageModel

from src.generation_utils import generate
from src.prompts import EVAL_QUESTIONS, SYSTEM_PROMPT

FastLanguageModel.for_inference(model)

for question in EVAL_QUESTIONS:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    answer = generate(model, tokenizer, prompt, max_new_tokens=200, temperature=0.7)
    print(f"Q: {question}\nA: {answer}\n{'-' * 80}")